# Инференс всех 5 адаптеров на BIRD card_games

Генерирует SQL-предсказания для каждого FT-адаптера на 31 реальном вопросе BIRD.
Сохраняет `eval_predictions.json` в папку каждого эксперимента на Drive.

**Важно**: `max_seq_length=12288` — как при обучении, иначе обрезка промпта.

**Запуск**: ячейка 0 (install) → Перезапустить сеанс → остальные по порядку.

После — скачать 5 `eval_predictions.json` локально и запустить `scripts/compare_generators.py`.

In [ ]:
# 0. Установка (после неё — Среда выполнения → Перезапустить сеанс)
!pip install -q --upgrade pip
!pip install -q --upgrade unsloth unsloth_zoo
print('✓ install done — ПЕРЕЗАПУСТИ СЕАНС, потом ячейка 1')

## 1. Импорты + Drive + данные

In [ ]:
from unsloth import FastLanguageModel
import torch, json, gc
from peft import PeftModel

from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/text2sql'
MAX_SEQ_LEN = 12288
BASE_MODEL = 'unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit'

# Имена экспериментов — из реестра (единый источник правды).
reg = json.load(open(f'{BASE}/registry.json'))
EXPERIMENTS = [e['name'] for e in reg['experiments']]

# BIRD card_games subset — положи bird_large.json в MyDrive/text2sql/
bird = json.load(open(f'{BASE}/bird_large.json'))
questions = [q for q in bird if q['db_id'] == 'card_games']
print(f'✓ torch {torch.__version__} | вопросов: {len(questions)} | experiments: {EXPERIMENTS}')

## 2. Системный промпт (из train.jsonl) + SQL-шаблон

In [ ]:
# Системный промпт идентичен для всех (одна БД). Берём из любого train.jsonl.
import glob
def find_system_prompt():
    cands = [f'{BASE}/{e}/train.jsonl' for e in EXPERIMENTS] \
          + glob.glob(f'{BASE}/**/train.jsonl', recursive=True)
    for p in cands:
        try:
            sp = json.loads(open(p).readline())['messages'][0]['content']
            print(f'  взят из: {p}')
            return sp
        except Exception:
            continue
    raise FileNotFoundError('train.jsonl не найден на Drive')

SYSTEM_PROMPT = find_system_prompt()
print(f'✓ system prompt: {len(SYSTEM_PROMPT)} символов')

# Передавать evidence в вопрос (снимает структурный потолок BIRD ~20%)
USE_EVIDENCE = True

SQL_PROMPT_TEMPLATE = '''
Преобразуй следующий запрос в SQL:
Запрос: {user_query}

Требования:
1. Только SQL, диалект {sql_dialect} - совместимый синтаксис
2. Обязательные комментарии перед запросом
3. Четкое соответствие запросу
4. Используй LIKE, если запрос касается имён собственных, фамилий, названий подразделений.
Всегда используй LIKE, если запрос касается значений из таблицы, которых нет явно
в промпте. В таблице слова могут быть в разных падежах, с кавычками или без,
ты должен УЧИТЫВАТЬ ЭТО!
5. Правила использования DISTINCT:
   - DISTINCT пишется ТОЛЬКО сразу после SELECT: "SELECT DISTINCT col" — правильно.
   - НИКОГДА не пиши DISTINCT после запятой: "SELECT col1, DISTINCT col2" — НЕВАЛИДНЫЙ SQL!
   - Если нужно SELECT DISTINCT с ORDER BY — все колонки ORDER BY ОБЯЗАНЫ быть в SELECT.
   - Если сортируешь по вычисляемому выражению, которого нет в SELECT — используй GROUP BY
     вместо DISTINCT, или включи выражение в SELECT.
НЕ ОФОРМЛЯЙ НИКАК СВОЙ ОТВЕТ. ТВОЙ ОТВЕТ - ТОЛЬКО ЗАПРОС, НИЧЕГО БОЛЕЕ!

SQL запрос:'''

## 3. Инференс всех 5 адаптеров

In [ ]:
import os

SKIP_IF_EXISTS = True  # пропускать если eval_predictions.json уже есть

def run_inference(model, tok, tag):
    """Прогнать загруженную модель по всем вопросам, сохранить predictions."""
    FastLanguageModel.for_inference(model)
    preds = []
    for i, q in enumerate(questions):
        question = q['question']
        if USE_EVIDENCE and q.get('evidence'):
            question = f"{q['question']} (подсказка: {q['evidence']})"
        user_msg = SQL_PROMPT_TEMPLATE.format(
            user_query=question, sql_dialect='PostgreSQL')
        msgs = [{'role':'system','content':SYSTEM_PROMPT},
                {'role':'user','content':user_msg}]
        text = tok.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)
        enc = tok(text, return_tensors='pt', truncation=True,
                  max_length=MAX_SEQ_LEN).to('cuda')
        out = model.generate(
            input_ids=enc['input_ids'], attention_mask=enc['attention_mask'],
            max_new_tokens=512, do_sample=False, pad_token_id=tok.eos_token_id)
        sql = tok.decode(out[0][enc['input_ids'].shape[1]:],
                         skip_special_tokens=True).strip()
        sql = sql.replace('```sql','').replace('```','').strip()
        preds.append({'question_id': str(q['question_id']),
                      'question': q['question'], 'gold_sql': q['SQL'],
                      'predicted_sql': sql,
                      'difficulty': q.get('difficulty','unknown')})
        if i < 2: print(f'  Q: {q["question"][:45]}\n  A: {sql[:70]}')
    os.makedirs(f'{BASE}/{tag}', exist_ok=True)
    json.dump(preds, open(f'{BASE}/{tag}/eval_predictions.json','w'),
              ensure_ascii=False, indent=2)
    print(f'✓ {len(preds)} → {tag}/eval_predictions.json')

def eval_one(tag, adapter_path=None):
    """Прогнать один эксперимент: baseline (adapter=None) или с адаптером."""
    pred_path = f'{BASE}/{tag}/eval_predictions.json'
    if SKIP_IF_EXISTS and os.path.exists(pred_path):
        print(f'⏭  {tag}: predictions уже есть в {pred_path} — пропускаю')
        return
    print(f'\n{"="*55}\n{tag}\n{"="*55}')
    try:
        model, tok = FastLanguageModel.from_pretrained(
            BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True)
        if adapter_path is not None:
            model = PeftModel.from_pretrained(model, adapter_path)
        run_inference(model, tok, tag)
    except Exception as e:
        print(f'✗ {tag} УПАЛ: {e}')
    finally:
        torch.cuda.empty_cache(); gc.collect()

## 4. По одной ячейке на eval (baseline + 5 адаптеров)

Запускай нужное по мере готовности адаптеров. Можно остановиться после baseline + gpt41_bi для быстрого результата (2 строки в таблице).

Каждая занимает ~3 минуты (загрузка + 31 вопрос инференса).

In [ ]:
# 🔵 BASELINE — базовый Qwen БЕЗ адаптера (точка отсчёта)
eval_one('baseline', adapter_path=None)

In [ ]:
# 1️⃣ gpt41_bi — приоритетный
eval_one('gpt41_bi', adapter_path=f'{BASE}/gpt41_bi/adapter')

In [ ]:
# 2️⃣ codexmini_bi
eval_one('codexmini_bi', adapter_path=f'{BASE}/codexmini_bi/adapter')

In [ ]:
# 3️⃣ gpt41nano_bi
eval_one('gpt41nano_bi', adapter_path=f'{BASE}/gpt41nano_bi/adapter')

In [ ]:
# 4️⃣ gpt4omini_bi
eval_one('gpt4omini_bi', adapter_path=f'{BASE}/gpt4omini_bi/adapter')

In [ ]:
# 5️⃣ gpt41mini_bi
eval_one('gpt41mini_bi', adapter_path=f'{BASE}/gpt41mini_bi/adapter')